# 03c: VADER Baseline vs BART-MNLI — Labeling Method Evaluation

**Author:** Member 5

This notebook adds VADER as a lightweight, training-free baseline for business urgency
labelling and compares it against the BART-MNLI zero-shot model from 03b.
Both methods map ticket text to the same three labels (High Urgency / Low Urgency / Churn Risk),
enabling a direct comparison across three evaluation dimensions (4a–4c).

**Hypotheses**
- **H1:** VADER compound scores will broadly agree with BART-MNLI on clear-cut cases
  (strongly negative text → High Urgency) but diverge on ambiguous tickets,
  yielding moderate Cohen's Kappa (0.2–0.5).
- **H2:** Both methods should show non-uniform urgency distributions across NMF topics —
  account-access and data-loss topics should skew toward High Urgency regardless of method.
- **H3:** BART-MNLI will align more closely with the initial k-means clusters than VADER,
  because it captures semantic context rather than surface-level sentiment polarity.

---
## Section 0: Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.metrics import (
    cohen_kappa_score, adjusted_rand_score,
    confusion_matrix, accuracy_score, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
nltk.download('vader_lexicon', quiet=True)

root        = Path.cwd().parent.parent
results_dir = root / 'results' / 'result_03_Topic_and_Insight'
data_path   = root / 'notebooks' / '02_Representation_Experiments' / 'kaggle_data' / 'customer_support_tickets.csv'
bart_path   = results_dir / '03b_labeled_data.csv'

results_dir.mkdir(parents=True, exist_ok=True)
print(f'Raw data found  : {data_path.exists()}')
print(f'03b output found: {bart_path.exists()}')

---
## Section 1: Load Raw Ticket Data

In [ ]:
raw = pd.read_csv(data_path)
raw['ticket_id'] = raw['Ticket ID'].astype(int).apply(lambda x: f'TICKET_{x-1:08d}')

df = raw[['ticket_id', 'Ticket Description', 'Ticket Type', 'Ticket Priority']].copy()
df = df.rename(columns={'Ticket Description': 'description'})
df['description'] = df['description'].astype(str).str.strip()
df = df[df['description'] != ''].reset_index(drop=True)

print(f'Tickets loaded: {len(df)}')
print('\nTicket Priority distribution:')
print(df['Ticket Priority'].value_counts())

---
## Section 2: VADER Baseline

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_label(compound):
    # thresholds tuned to approximate BART's class split (~75% High, ~24% Low, ~1% Churn)
    if compound <= -0.5:
        return 'Churn Risk'
    elif compound < 0.0:
        return 'High Urgency'
    else:
        return 'Low Urgency'

df['vader_compound'] = df['description'].apply(
    lambda x: sia.polarity_scores(str(x)[:2000])['compound']
)
df['vader_label'] = df['vader_compound'].apply(get_label)

print('VADER label distribution:')
print(df['vader_label'].value_counts())
print('\nCompound score summary:')
print(df['vader_compound'].describe().round(3))

save_path = results_dir / '03c_vader_predictions.csv'
df[['ticket_id', 'vader_compound', 'vader_label']].to_csv(save_path, index=False)
print(f'\nSaved: {save_path}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['vader_compound'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(-0.5, color='crimson', linestyle='--', label='Churn Risk threshold')
axes[0].axvline(0.0,  color='orange',  linestyle='--', label='High/Low boundary')
axes[0].set(xlabel='VADER Compound Score', ylabel='Count', title='Compound Score Distribution')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

counts = df['vader_label'].value_counts().reindex(
    ['High Urgency', 'Low Urgency', 'Churn Risk']
).dropna()
axes[1].bar(counts.index, counts.values,
            color=['#d62728', '#1f77b4', '#ff7f0e'], edgecolor='white')
axes[1].set(ylabel='Ticket Count', title='VADER Label Distribution')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / '03c_vader_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Section 3: Load BART-MNLI Results from 03b

> `03b_labeled_data.csv` is git-ignored and produced by Member C (notebook 03b).  
> Place it at `results/result_03_Topic_and_Insight/03b_labeled_data.csv` before running Sections 4–6.  
> Each comparison cell will skip gracefully if the file is absent.

In [ ]:
bart_loaded = bart_path.exists()

if bart_loaded:
    bart_df = pd.read_csv(bart_path)
    keep = [c for c in ['ticket_id', 'business_label', 'business_confidence',
                         'nmf_dominant_topic', 'axis1_preferred_cluster']
            if c in bart_df.columns]
    bart_df = bart_df[keep].rename(columns={
        'business_label':      'bart_label',
        'business_confidence': 'bart_conf'
    })
    merged = df.merge(bart_df, on='ticket_id', how='inner')
    print(f'Merged {len(merged)} tickets')
    print('\nBART label distribution:')
    print(merged['bart_label'].value_counts())
    print('\nVADER label distribution (merged set):')
    print(merged['vader_label'].value_counts())
else:
    print('03b_labeled_data.csv not found.')
    print(f'Expected path: {bart_path}')

---
## Section 4 (4a): Consistency and Robustness Across Methods

In [ ]:
if not bart_loaded:
    print('Skipping — 03b data not available.')
else:
    # drop the small number of BART 'Uncategorized' rows before computing agreement
    ev = merged[merged['bart_label'] != 'Uncategorized'].copy()

    agree = (ev['vader_label'] == ev['bart_label']).mean()
    kappa = cohen_kappa_score(ev['vader_label'], ev['bart_label'])

    print(f'Evaluated on {len(ev)} tickets (Uncategorized rows excluded)')
    print(f'\nAgreement rate : {agree:.3f}  ({agree*100:.1f}%)')
    print(f'Kappa          : {kappa:.3f}')
    print('(Kappa guide: <0.2 slight, 0.2-0.4 fair, 0.4-0.6 moderate, >0.6 substantial)')

In [ ]:
if not bart_loaded:
    print('Skipping.')
else:
    label_order = [l for l in ['High Urgency', 'Low Urgency', 'Churn Risk']
                   if l in ev['vader_label'].values or l in ev['bart_label'].values]
    cm = confusion_matrix(ev['vader_label'], ev['bart_label'], labels=label_order)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_order, yticklabels=label_order, ax=ax)
    ax.set(xlabel='BART-MNLI', ylabel='VADER',
           title=f'Agreement Matrix  (agree={agree:.1%}, kappa={kappa:.3f})')
    plt.tight_layout()
    plt.savefig(results_dir / '03c_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# robustness check: if BART urgency labels are valid, High Urgency tickets should
# have lower VADER compound scores than Low Urgency ones
if not bart_loaded:
    print('Skipping.')
else:
    plot_order = [l for l in ['High Urgency', 'Low Urgency', 'Churn Risk']
                  if l in ev['bart_label'].values]

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.violinplot(data=ev, x='bart_label', y='vader_compound',
                   order=plot_order, ax=ax, inner='median')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set(xlabel='BART-MNLI Label', ylabel='VADER Compound Score',
           title='VADER Score Distribution by BART Label')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / '03c_robustness_violin.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('Mean VADER compound per BART label:')
    for lbl in plot_order:
        m = ev.loc[ev['bart_label'] == lbl, 'vader_compound'].mean()
        print(f'  {lbl:<18}: {m:+.3f}')

---
## Section 5 (4b): Coherence and Interpretability of Issue Categories

In [ ]:
if not bart_loaded:
    print('Skipping — 03b data not available.')
else:
    def top_words(texts, n=12):
        if len(texts) < 2:
            return []
        vec = TfidfVectorizer(max_features=3000, stop_words='english', min_df=2)
        try:
            mat = vec.fit_transform(texts)
            scores = mat.mean(axis=0).A1
            idx = scores.argsort()[::-1][:n]
            vocab = vec.get_feature_names_out()
            return [(vocab[i], float(scores[i])) for i in idx]
        except ValueError:
            return []

    focus  = ['High Urgency', 'Low Urgency']
    methods = [('VADER', 'vader_label'), ('BART-MNLI', 'bart_label')]
    colors  = {'High Urgency': '#d62728', 'Low Urgency': '#1f77b4'}

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    for row, label in enumerate(focus):
        for col, (name, col_name) in enumerate(methods):
            texts = merged.loc[merged[col_name] == label, 'description'].tolist()
            words = top_words(texts, n=12)
            ax = axes[row][col]
            if words:
                ws, sc = zip(*words)
                ax.barh(range(len(ws)), sc, color=colors[label], alpha=0.8, edgecolor='white')
                ax.set_yticks(range(len(ws)))
                ax.set_yticklabels(ws, fontsize=9)
                ax.invert_yaxis()
            ax.set_title(f'{name} — {label}  (n={len(texts)})', fontsize=10)
            ax.set_xlabel('Mean TF-IDF', fontsize=8)
            ax.grid(axis='x', alpha=0.3)

            top6 = ', '.join(w for w, _ in words[:6])
            print(f'[{name}] {label}: {top6}')

    plt.suptitle('Top TF-IDF Terms per Urgency Label per Method', fontsize=12)
    plt.tight_layout()
    plt.savefig(results_dir / '03c_coherence_top_words.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if not bart_loaded:
    print('Skipping.')
else:
    label_order = ['High Urgency', 'Low Urgency', 'Churn Risk']
    x = np.arange(len(label_order))
    w = 0.35

    v_counts = [merged['vader_label'].value_counts().get(l, 0) for l in label_order]
    b_counts = [merged['bart_label'].value_counts().get(l, 0)  for l in label_order]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - w/2, v_counts, w, label='VADER',     color='steelblue',  edgecolor='white')
    ax.bar(x + w/2, b_counts, w, label='BART-MNLI', color='darkorange', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(label_order)
    ax.set(ylabel='Ticket Count', title='Label Distribution: VADER vs BART-MNLI')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / '03c_label_distribution_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # a coherent labeling should be monotonically ordered: Low Urgency > High Urgency > Churn Risk
    print('Mean VADER compound per label per method:')
    for name, col_name in methods:
        print(f'  {name}:')
        for lbl in label_order:
            if lbl in merged[col_name].values:
                m = merged.loc[merged[col_name] == lbl, 'vader_compound'].mean()
                n = (merged[col_name] == lbl).sum()
                print(f'    {lbl:<20}: {m:+.3f}  (n={n})')

---
## Section 6 (4c): Alignment with Initial Clustering Analysis

In [ ]:
if not bart_loaded:
    print('Skipping — 03b data not available.')
else:
    print('Adjusted Rand Index (higher = stronger alignment with unsupervised structure)\n')

    ari_v_cl = ari_b_cl = None
    if 'axis1_preferred_cluster' in merged.columns:
        clusters = merged['axis1_preferred_cluster'].astype(int).values
        ari_v_cl = adjusted_rand_score(clusters, merged['vader_label'])
        ari_b_cl = adjusted_rand_score(clusters, merged['bart_label'])
        print(f'ARI vs k-Means clusters (Notebook 01):')
        print(f'  VADER    : {ari_v_cl:.4f}')
        print(f'  BART-MNLI: {ari_b_cl:.4f}')
    else:
        print('axis1_preferred_cluster not in 03b output — cluster ARI skipped.')

    ari_v_nmf = ari_b_nmf = None
    if 'nmf_dominant_topic' in merged.columns:
        nmf = merged['nmf_dominant_topic'].astype(int).values
        ari_v_nmf = adjusted_rand_score(nmf, merged['vader_label'])
        ari_b_nmf = adjusted_rand_score(nmf, merged['bart_label'])
        print(f'\nARI vs NMF topics (Notebook 03a):')
        print(f'  VADER    : {ari_v_nmf:.4f}')
        print(f'  BART-MNLI: {ari_b_nmf:.4f}')
    else:
        print('nmf_dominant_topic not in 03b output — NMF ARI skipped.')

In [ ]:
if not bart_loaded:
    print('Skipping.')
elif 'nmf_dominant_topic' not in merged.columns:
    print('nmf_dominant_topic not available — heatmap skipped.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, (name, col_name) in zip(axes, [('VADER', 'vader_label'), ('BART-MNLI', 'bart_label')]):
        ct = pd.crosstab(merged['nmf_dominant_topic'], merged[col_name], normalize='index')
        ct = ct[[c for c in ['High Urgency', 'Low Urgency', 'Churn Risk'] if c in ct.columns]]
        ct.index = [f'Topic {i}' for i in ct.index]
        sns.heatmap(ct, annot=True, fmt='.2f', cmap='YlOrRd',
                    vmin=0, vmax=1, linewidths=0.5, ax=ax)
        ax.set(title=f'{name} — urgency proportion per NMF topic',
               xlabel='Urgency Label', ylabel='NMF Topic')

    plt.suptitle('Urgency Distribution per NMF Topic (row-normalised)', fontsize=12)
    plt.tight_layout()
    plt.savefig(results_dir / '03c_alignment_nmf_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# proxy evaluation: use the dataset's own Ticket Priority as an approximate ground truth
if not bart_loaded:
    print('Skipping.')
else:
    priority_map = {
        'Critical': 'High Urgency', 'High':   'High Urgency',
        'Medium':   'Low Urgency',  'Low':    'Low Urgency'
    }
    merged['priority_proxy'] = merged['Ticket Priority'].map(priority_map)
    prox = merged[merged['priority_proxy'].notna()].copy()

    # fold minority classes into High Urgency for binary evaluation
    v_pred = prox['vader_label'].replace({'Churn Risk': 'High Urgency'})
    b_pred = prox['bart_label'].replace({'Churn Risk': 'High Urgency',
                                         'Uncategorized': 'High Urgency'})

    vader_acc = accuracy_score(prox['priority_proxy'], v_pred)
    bart_acc  = accuracy_score(prox['priority_proxy'], b_pred)
    vader_f1  = f1_score(prox['priority_proxy'], v_pred, average='weighted', zero_division=0)
    bart_f1   = f1_score(prox['priority_proxy'], b_pred, average='weighted', zero_division=0)

    print('Proxy evaluation against Ticket Priority:')
    print(f'  {"Method":<12} {"Accuracy":>10} {"F1 (weighted)":>15}')
    print(f'  {"VADER":<12} {vader_acc:>10.3f} {vader_f1:>15.3f}')
    print(f'  {"BART-MNLI":<12} {bart_acc:>10.3f} {bart_f1:>15.3f}')

---
## Section 7: Summary

In [ ]:
print('=' * 60)
print('03c SUMMARY — VADER vs BART-MNLI')
print('=' * 60)

if not bart_loaded:
    print('Full summary requires 03b_labeled_data.csv (see Section 3).')
    print(f'VADER predictions saved: {results_dir / "03c_vader_predictions.csv"}')
else:
    rows = [
        ('4a', 'Agreement Rate',     f'{agree:.3f}',  '—'),
        ('4a', 'Kappa',              f'{kappa:.3f}',  '—'),
    ]
    if ari_v_cl is not None:
        rows.append(('4c', 'ARI vs k-Means', f'{ari_v_cl:.3f}', f'{ari_b_cl:.3f}'))
    if ari_v_nmf is not None:
        rows.append(('4c', 'ARI vs NMF',     f'{ari_v_nmf:.3f}', f'{ari_b_nmf:.3f}'))
    rows += [
        ('4c', 'Proxy Accuracy',     f'{vader_acc:.3f}', f'{bart_acc:.3f}'),
        ('4c', 'Proxy F1 (weighted)', f'{vader_f1:.3f}',  f'{bart_f1:.3f}'),
    ]

    summary = pd.DataFrame(rows, columns=['Section', 'Metric', 'VADER', 'BART-MNLI'])
    print(summary.to_string(index=False))

    summary.to_csv(results_dir / '03c_evaluation_summary.csv', index=False)
    print(f'\nSaved: {results_dir / "03c_evaluation_summary.csv"}')

print('=' * 60)